### Data Processing

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import precision_score
from sklearn.metrics import classification_report
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix
from sklearn.inspection import permutation_importance
!pip install mlxtend
from mlxtend.plotting import plot_decision_regions 
from sklearn.ensemble import RandomForestClassifier


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
seattle = pd.read_csv("listings.csv.gz")
seattle.columns

Index(['id', 'listing_url', 'scrape_id', 'last_scraped', 'source', 'name',
       'description', 'neighborhood_overview', 'picture_url', 'host_id',
       'host_url', 'host_name', 'host_since', 'host_location', 'host_about',
       'host_response_time', 'host_response_rate', 'host_acceptance_rate',
       'host_is_superhost', 'host_thumbnail_url', 'host_picture_url',
       'host_neighbourhood', 'host_listings_count',
       'host_total_listings_count', 'host_verifications',
       'host_has_profile_pic', 'host_identity_verified', 'neighbourhood',
       'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude',
       'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms',
       'bathrooms_text', 'bedrooms', 'beds', 'amenities', 'price',
       'minimum_nights', 'maximum_nights', 'minimum_minimum_nights',
       'maximum_minimum_nights', 'minimum_maximum_nights',
       'maximum_maximum_nights', 'minimum_nights_avg_ntm',
       'maximum_nights_avg_ntm', 'ca

In [3]:
# turn price into string
seattle["price"] = (
    seattle["price"]
    .replace(r"[\$,]", "", regex=True)
    .astype(float)
)

In [4]:
# fill null with 0
seattle = seattle.fillna(0)
# filter price <= 0
seattle = seattle[seattle["price"] > 0]
# remove extreme price
p99_seattle = seattle["price"].quantile(0.99)
seattle = seattle[seattle["price"] <= p99_seattle]

In [5]:
# remove extreme value
p99_bedroom = seattle["bedrooms"].quantile(0.99)
p99_beds = seattle["beds"].quantile(0.99)
p99_bathroom = seattle["bathrooms"].quantile(0.99)
p99_accomodates = seattle["accommodates"].quantile(0.99)
seattle = seattle[seattle["bedrooms"] <= p99_bedroom]
seattle = seattle[seattle["beds"] <= p99_beds]
seattle = seattle[seattle["bathrooms"] <= p99_bathroom]
seattle = seattle[seattle["accommodates"] <= p99_accomodates]
seattle

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,...,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,6606,https://www.airbnb.com/rooms/6606,20250925032813,2025-09-25,city scrape,"Fab, private seattle urban cottage!","This tiny cottage is only 15x10, but it has ev...","A peaceful yet highly accessible neighborhood,...",https://a0.muscache.com/pictures/45742/21116d7...,14942,...,4.77,4.88,4.57,str-opli-19-002622,f,3,3,0,0,0.82
1,9419,https://www.airbnb.com/rooms/9419,20250925032813,2025-09-25,city scrape,Glorious sun room w/ memory foambed,This beautiful double room features sun filled...,"Lots of restaurants (see our guide book) bars,...",https://a0.muscache.com/pictures/56645186/e5fb...,30559,...,4.89,4.70,4.69,Exempt,f,10,0,10,0,1.19
3,11012,https://www.airbnb.com/rooms/11012,20250925032813,2025-09-25,city scrape,"the orange house, quiet 'n central",0,0,https://a0.muscache.com/pictures/682034/54bc27...,14942,...,4.72,4.86,4.74,str-opli-19-002622,f,3,3,0,0,0.51
4,25002,https://www.airbnb.com/rooms/25002,20250925032813,2025-09-25,city scrape,Beautiful Private Spot in North Ballard,"-Great eating , Delancey, Fat Hen, 3 blocks aw...",Great walking neighborhood! We are in between...,https://a0.muscache.com/pictures/491561/cf5270...,102684,...,4.98,4.90,4.90,STR-OPLI-19-002617,t,1,1,0,0,6.06
5,26795,https://www.airbnb.com/rooms/26795,20250925032813,2025-09-25,city scrape,Lake Union Cottage - Shore and City View,"This sunny, corner lot is directly across from...",This area of the Eastlake Neighborhood is quie...,https://a0.muscache.com/pictures/179416/54927c...,114228,...,4.58,4.83,4.36,0,f,1,1,0,0,0.35
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6991,1515445606707131986,https://www.airbnb.com/rooms/1515445606707131986,20250925032813,2025-09-25,city scrape,Centrally located private room in shoreline-2,"Private room and bathroom in a townhome, centr...",0,https://a0.muscache.com/pictures/hosting/Hosti...,127525192,...,0.00,0.00,0.00,STR-OPLI-25-002944,f,3,0,3,0,0.00
6992,1515452490876536616,https://www.airbnb.com/rooms/1515452490876536616,20250925032813,2025-09-25,city scrape,Centrally located private room in Shoreline-3,"Private room and bathroom in a townhome, centr...",0,https://a0.muscache.com/pictures/hosting/Hosti...,127525192,...,0.00,0.00,0.00,STR-OPLI-25-002944,f,3,0,3,0,0.00
6993,1516340815301554136,https://www.airbnb.com/rooms/1516340815301554136,20250925032813,2025-09-25,city scrape,"Top floor oasis, comfortable and elegant.",This oasis is a perfect place to recover from ...,0,https://a0.muscache.com/pictures/hosting/Hosti...,308306114,...,0.00,0.00,0.00,STR-OPLI-19-803339,f,2,1,1,0,0.00
6994,1516722602078881683,https://www.airbnb.com/rooms/1516722602078881683,20250925032813,2025-09-25,city scrape,"Blueground | West Queen Anne, view, nr Kerry Park","Discover the best of Seattle, with this one be...",0,https://a0.muscache.com/pictures/prohost-api/H...,107434423,...,0.00,0.00,0.00,0,t,344,344,0,0,0.00


In [6]:
%cd imt574_ml_final_project
%pwd

/Users/peiyingw/imt574_ml_final_project


'/Users/peiyingw/imt574_ml_final_project'

In [109]:
# turning pandas to csv 
seattle.to_csv('seattle_filtered.csv', index=False)